<a href="https://colab.research.google.com/github/SttApollo/SttApollo/blob/main/UploadLocalDoc_DoItYourSelf_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')
!pip install -q langchain langchain_community langchain_text_splitters sentence_transformers chromadb langchain_huggingface
!pip install langchain-groq
!pip install -q unstructured[docx] python-docx
!pip install -q python-magic
print("installation complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 k

In [ ]:
!pip install -q langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFaceHub

print(f"Import complete")

Import complete


In [ ]:
"""
═══════════════════════════════════════════════════════════════
UPLOAD DOCUMENTS WITH VALIDATION
Upload from desktop with format and size checking
═══════════════════════════════════════════════════════════════
"""

from google.colab import files
import os

print("📂 UPLOAD DOCUMENTS FROM YOUR COMPUTER")
print("=" * 70)

# Settings
ALLOWED_EXTENSIONS = ['.pdf', '.txt', '.docx', '.md', '.doc']
MAX_FILE_SIZE_MB = 5
MAX_TOTAL_SIZE_MB = 25

print(f"\n📋 Requirements:")
print(f"   • Allowed formats: {', '.join(ALLOWED_EXTENSIONS)}")
print(f"   • Max file size: {MAX_FILE_SIZE_MB} MB")
print(f"   • Max total size: {MAX_TOTAL_SIZE_MB} MB")
print()

# Create destination directory
destination_folder = "Sample Documents"
os.makedirs(destination_folder, exist_ok=True)

# Upload files
print("⏳ Click 'Choose Files' to select documents...\n")
uploaded = files.upload()

print("\n" + "=" * 70)
print("VALIDATING UPLOADED FILES")
print("=" * 70)

files_copied = []
files_rejected = []
total_size = 0

for filename, content in uploaded.items():
    size_mb = len(content) / (1024 * 1024)
    ext = os.path.splitext(filename)[1].lower()

    # Validate extension
    if ext not in ALLOWED_EXTENSIONS:
        print(f"\n❌ {filename}")
        print(f"   Reason: Invalid format (must be {', '.join(ALLOWED_EXTENSIONS)})")
        files_rejected.append(filename)
        continue

    # Validate file size
    if size_mb > MAX_FILE_SIZE_MB:
        print(f"\n❌ {filename}")
        print(f"   Reason: File too large ({size_mb:.1f} MB, max {MAX_FILE_SIZE_MB} MB)")
        files_rejected.append(filename)
        continue

    # Check total size
    if (total_size + len(content)) / (1024 * 1024) > MAX_TOTAL_SIZE_MB:
        print(f"\n❌ {filename}")
        print(f"   Reason: Would exceed total size limit ({MAX_TOTAL_SIZE_MB} MB)")
        files_rejected.append(filename)
        continue

    # File is valid - save it
    dest_path = os.path.join(destination_folder, filename)

    with open(dest_path, 'wb') as f:
        f.write(content)

    total_size += len(content)
    size_kb = len(content) / 1024

    # Determine emoji
    emoji = {
        '.pdf': '📄',
        '.txt': '📝',
        '.md': '📝',
        '.doc': '📘',
        '.docx': '📘',
    }.get(ext, '📎')

    print(f"\n✅ {emoji} {filename}")
    print(f"   Size: {size_kb:.1f} KB")
    print(f"   Format: {ext}")

    files_copied.append(dest_path)

# Summary
print("\n" + "=" * 70)
print("UPLOAD SUMMARY")
print("=" * 70)
print(f"✅ Files uploaded: {len(files_copied)}")

if files_rejected:
    print(f"❌ Files rejected: {len(files_rejected)}")
    print(f"   Rejected: {', '.join(files_rejected)}")

print(f"📊 Total size: {total_size / 1024:.1f} KB")
print(f"📁 Location: {destination_folder}/")

if files_copied:
    print("\n📋 Files ready for processing:")
    for filepath in files_copied:
        print(f"   • {os.path.basename(filepath)}")

    print("\n✅ Ready to build RAG system!")
else:
    print("\n⚠️  No valid files uploaded. Please try again.")

📂 UPLOAD DOCUMENTS FROM YOUR COMPUTER

📋 Requirements:
   • Allowed formats: .pdf, .txt, .docx, .md, .doc
   • Max file size: 5 MB
   • Max total size: 25 MB

⏳ Click 'Choose Files' to select documents...



Saving API Documentation - CloudSync REST API v3.2.docx to API Documentation - CloudSync REST API v3.2.docx
Saving TechCorp Employee Handbook 2025.docx to TechCorp Employee Handbook 2025.docx

VALIDATING UPLOADED FILES

✅ 📘 API Documentation - CloudSync REST API v3.2.docx
   Size: 223.6 KB
   Format: .docx

✅ 📘 TechCorp Employee Handbook 2025.docx
   Size: 1650.6 KB
   Format: .docx

UPLOAD SUMMARY
✅ Files uploaded: 2
📊 Total size: 1874.3 KB
📁 Location: Sample Documents/

📋 Files ready for processing:
   • API Documentation - CloudSync REST API v3.2.docx
   • TechCorp Employee Handbook 2025.docx

✅ Ready to build RAG system!


In [ ]:
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    UnstructuredWordDocumentLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob
import os

print("📄 LOADING AND CHUNKING DOCUMENTS")
print("=" * 70)

# Load all documents from my_documents folder
documents = []
document_folder = "Sample Documents"

print("\n⏳ Loading documents...\n")

for filepath in glob.glob(f"{document_folder}/*"):
    filename = os.path.basename(filepath)

    try:
        # Determine loader based on file extension
        if filepath.endswith('.txt') or filepath.endswith('.md'):
            loader = TextLoader(filepath)
        elif filepath.endswith('.pdf'):
            loader = PyPDFLoader(filepath)
        elif filepath.endswith('.docx') or filepath.endswith('.doc'):
            loader = UnstructuredWordDocumentLoader(filepath)
        else:
            print(f"⊝ {filename} - Unsupported format, skipping")
            continue

        # Load document
        docs = loader.load()
        documents.extend(docs)

        # Show what was loaded
        total_chars = sum(len(doc.page_content) for doc in docs)
        print(f"✓ {filename}")
        print(f"   Pages/Sections: {len(docs)}")
        print(f"   Characters: {total_chars:,}")
        print()

    except Exception as e:
        print(f"❌ {filename} - Error: {str(e)}\n")

print("-" * 70)
print(f"✅ Loaded {len(documents)} document(s)")
print(f"📊 Total characters: {sum(len(doc.page_content) for doc in documents):,}")

# Chunk the documents
print("\n✂️  CHUNKING DOCUMENTS")
print("=" * 70)
print("\nChunk settings:")
print("   • Chunk size: 500 characters")
print("   • Overlap: 50 characters")
print("   • Splits at: paragraphs → sentences → words")
print()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Created {len(chunks)} chunks")

# Show sample chunk
print("\n📋 Sample chunk:")
print("-" * 70)
print(chunks[0].page_content[:300])
if len(chunks[0].page_content) > 300:
    print("...")
print("-" * 70)
print(f"Chunk length: {len(chunks[0].page_content)} characters")

# Chunk statistics
chunk_sizes = [len(chunk.page_content) for chunk in chunks]
avg_size = sum(chunk_sizes) / len(chunk_sizes)
min_size = min(chunk_sizes)
max_size = max(chunk_sizes)

print(f"\n📊 Chunk Statistics:")
print(f"   Total chunks: {len(chunks)}")
print(f"   Average size: {avg_size:.0f} characters")
print(f"   Smallest: {min_size} characters")
print(f"   Largest: {max_size} characters")

print("\n" + "=" * 70)
print("✅ CHUNKING COMPLETE")
print("=" * 70)

📄 LOADING AND CHUNKING DOCUMENTS

⏳ Loading documents...

✓ TechCorp Employee Handbook 2025.docx
   Pages/Sections: 1
   Characters: 16,505

✓ API Documentation - CloudSync REST API v3.2.docx
   Pages/Sections: 1
   Characters: 19,162

----------------------------------------------------------------------
✅ Loaded 2 document(s)
📊 Total characters: 35,667

✂️  CHUNKING DOCUMENTS

Chunk settings:
   • Chunk size: 500 characters
   • Overlap: 50 characters
   • Splits at: paragraphs → sentences → words

✅ Created 93 chunks

📋 Sample chunk:
----------------------------------------------------------------------
Sample Document: TechCorp Employee Handbook 2025

Welcome to TechCorp

Welcome to TechCorp! We're thrilled to have you as part of our team. This handbook provides important information about our company policies, benefits, and workplace guidelines. Please take the time to read through this document 
...
----------------------------------------------------------------------
Chunk leng

In [ ]:

# ═══════════════════════════════════════════════════════════════
# STEP 2: LOAD EMBEDDING MODEL
# ═══════════════════════════════════════════════════════════════

from langchain_huggingface import HuggingFaceEmbeddings

print("STEP 2: LOADING EMBEDDING MODEL")
print("=" * 70)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

print("✅ Embedding model loaded (384 dimensions)\n")

# ═══════════════════════════════════════════════════════════════
# STEP 3: CREATE VECTOR DATABASE
# ═══════════════════════════════════════════════════════════════

from langchain_community.vectorstores import Chroma

print("STEP 3: BUILDING VECTOR DATABASE")
print("=" * 70)
print("⏳ Creating vector store (1-2 minutes)...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_documents"
)

print(f"✅ Vector database created!")
print(f"   Stored: {vectorstore._collection.count()} chunks")
print(f"   Collection: my_documents")

# Test query
test_results = vectorstore.similarity_search("How many holidays can i take", k=2)
print(f"\n🧪 Test query successful!")
print(f"   Sample result: {test_results[0].page_content[:100]}...\n")
print(f"   Sample result: {test_results[1].page_content[:100]}...\n")

print("=" * 70)
print("🎉 SETUP COMPLETE - READY TO QUERY!")
print("=" * 70)

STEP 2: LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded (384 dimensions)

STEP 3: BUILDING VECTOR DATABASE
⏳ Creating vector store (1-2 minutes)...

✅ Vector database created!
   Stored: 93 chunks
   Collection: my_documents

🧪 Test query successful!
   Sample result: Sick Leave: Employees accrue 10 sick days per year, available from day one. Sick leave can be used f...

   Sample result: Personal Days: All employees receive 3 personal days per year that can be used for any reason—no exp...

🎉 SETUP COMPLETE - READY TO QUERY!


In [ ]:
"""
═══════════════════════════════════════════════════════════════
STEP 5: SETUP LLM (Language Model)
Connect to Groq for text generation
═══════════════════════════════════════════════════════════════
"""

from langchain_groq import ChatGroq

print("🤖 SETTING UP LANGUAGE MODEL")
print("=" * 70)

from google.colab import userdata
# Retrieve the Groq API key from Colab's secrets manager
GROQ_API_KEY = userdata.get('Groq_KEY')

print("\n⏳ Connecting to Groq...")

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

print("✅ Connected to Groq successfully!")
print(f"\n📊 LLM Configuration:")
print(f"   Model: llama-3.3-70b-versatile")
print(f"   Temperature: 0 (deterministic)")
print(f"   Provider: Groq")

# Test the LLM
print("\n🧪 Testing LLM...")
test_response = llm.invoke("Say 'Hello, I am ready!'")

print(f"\n   Test prompt: 'Say Hello, I am ready!'")
print(f"   Response: {test_response.content}")

print("\n" + "=" * 70)
print("✅ LLM READY")
print("=" * 70)

🤖 SETTING UP LANGUAGE MODEL

⏳ Connecting to Groq...
✅ Connected to Groq successfully!

📊 LLM Configuration:
   Model: llama-3.3-70b-versatile
   Temperature: 0 (deterministic)
   Provider: Groq

🧪 Testing LLM...

   Test prompt: 'Say Hello, I am ready!'
   Response: Hello, I am ready!

✅ LLM READY


In [ ]:
"""
═══════════════════════════════════════════════════════════════
STEP 6: BUILD RAG FUNCTION
Combine retrieval and generation
═══════════════════════════════════════════════════════════════
"""

def ask_rag(question, k=3):
    """
    Query documents using RAG

    Args:
        question: The question to ask
        k: Number of relevant chunks to retrieve (default: 3)

    Returns:
        Dictionary with question, answer, and source documents
    """

    # Step 1: Retrieve relevant documents
    relevant_docs = vectorstore.similarity_search(question, k=k)

    # Step 2: Combine retrieved documents into context
    context = "\n\n".join([doc.page_content for doc in relevant_docs])

    # Step 3: Create augmented prompt
    prompt = f"""Answer the question based only on the following context:

Context:
{context}

Question: {question}

Answer:"""

    # Step 4: Generate answer using LLM
    response = llm.invoke(prompt)

    return {
        "question": question,
        "answer": response.content,
        "sources": relevant_docs,
        "num_sources": len(relevant_docs)
    }

print("🔧 RAG FUNCTION CREATED")
print("=" * 70)
print("\nFunction: ask_rag(question, k=3)")
print("\nWhat it does:")
print("   1. Retrieves relevant chunks from vector database")
print("   2. Combines chunks into context")
print("   3. Adds context to your question")
print("   4. Sends to LLM for answer generation")
print("   5. Returns answer with sources")

print("\n✅ RAG function ready to use!")
print("=" * 70)

🔧 RAG FUNCTION CREATED

Function: ask_rag(question, k=3)

What it does:
   1. Retrieves relevant chunks from vector database
   2. Combines chunks into context
   3. Adds context to your question
   4. Sends to LLM for answer generation
   5. Returns answer with sources

✅ RAG function ready to use!


In [ ]:
"""
═══════════════════════════════════════════════════════════════
STEP 7: TEST RAG SYSTEM
Query your documents!
═══════════════════════════════════════════════════════════════
"""

print("\n💬 TESTING RAG SYSTEM")
print("=" * 70)

# Test with sample questions
test_questions = [
    "What is the vacation policy?",
    "How many sick days do employees get?",
    "What health insurance options are available?",
]

print("\n🧪 Running test queries:\n")

for i, question in enumerate(test_questions, 1):
    print(f"{'=' * 70}")
    print(f"Question {i}: {question}")
    print(f"{'=' * 70}")

    # Get answer
    result = ask_rag(question, k=3)

    print(f"\n📝 Answer:")
    print(f"{result['answer']}")

    print(f"\n📚 Sources used: {result['num_sources']} chunks")
    print(f"\n📄 Source preview:")
    print(f"{result['sources'][0].page_content[:150]}...")

    print("\n")

print("=" * 70)
print("✅ RAG SYSTEM WORKING!")
print("=" * 70)


💬 TESTING RAG SYSTEM

🧪 Running test queries:

Question 1: What is the vacation policy?

📝 Answer:
The vacation policy is as follows: 

1. Vacation requests must be submitted through the HR portal at least two weeks in advance.
2. Managers approve requests based on business needs and team coverage.
3. Full-time employees accrue vacation days based on tenure: 
   - New employees accrue 15 days per year (1.25 days per month).
   - After 3 years of service, accrual increases to 20 days per year.
   - After 7 years, it increases to 25 days per year.
4. Part-time employees receive prorated vacation based on their scheduled hours.
5. Unused vacation days can be carried over to the next year, with a maximum carryover of 10 days.

📚 Sources used: 3 chunks

📄 Source preview:
Vacation requests should be submitted through the HR portal at least two weeks in advance. Managers will approve requests based on business needs and ...


Question 2: How many sick days do employees get?

📝 Answer:
10 sic

In [ ]:
"""
═══════════════════════════════════════════════════════════════
STEP 8: INTERACTIVE Q&A
Ask your own questions!
═══════════════════════════════════════════════════════════════
"""

print("\n🎮 INTERACTIVE Q&A MODE")
print("=" * 70)
print("\nAsk questions about your documents!")
print("Type 'exit' or 'quit' to stop\n")
print("-" * 70)

while True:
    # Get user question
    question = input("\n❓ Your question: ").strip()

    # Exit condition
    if question.lower() in ['exit', 'quit', '']:
        print("\n👋 Exiting Q&A mode. Thanks for trying RAG!")
        break

    print(f"\n⏳ Searching documents...")

    try:
        # Get answer
        result = ask_rag(question, k=3)

        print(f"\n" + "=" * 70)
        print(f"💡 ANSWER")
        print(f"=" * 70)
        print(f"\n{result['answer']}")

        print(f"\n" + "-" * 70)
        print(f"📚 Retrieved {result['num_sources']} relevant chunks")
        print(f"-" * 70)

        # Show sources
        print(f"\n📄 Top source:")
        print(f"{result['sources'][0].page_content[:200]}...")

    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        print("Try asking a different question.")

print("\n" + "=" * 70)


🎮 INTERACTIVE Q&A MODE

Ask questions about your documents!
Type 'exit' or 'quit' to stop

----------------------------------------------------------------------

❓ Your question: What is my remote work policy

⏳ Searching documents...

💡 ANSWER

As per the Remote Work Policy, you are eligible to work remotely up to 3 days per week, subject to manager approval and role requirements. To work remotely, you must:

1. Maintain a dedicated, professional workspace free from distractions
2. Be available during core business hours (10 AM - 4 PM)
3. Attend all required meetings via video conference

You are expected to maintain the same productivity and communication standards as those working in the office, and have a dedicated workspace with reliable internet connectivity. Remote work is a privilege, not a right, and may be revoked if performance or communication standards are not met.

----------------------------------------------------------------------
📚 Retrieved 3 relevant chunks
-----